[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/sandbox/run_ersilia_model_colab.ipynb)

# Run an Ersilia model in Colab

Pick a model, upload a file of SMILES, get predictions back. Three cells, in order.

Each cell is a form: you see the controls, not the code. Use *View -> Show code* if you want
to read what one does. The recipe behind them is the one established in
[`sif_feasibility_test.ipynb`](sif_feasibility_test.ipynb).


In [ ]:
#@title 1. Set up this runtime { display-mode: "form" }
#@markdown Installs Apptainer. Takes about 40 seconds, and is needed once per session.
import json
import os
import pathlib
import shutil
import subprocess
import time
import urllib.error
import urllib.request

BUCKET = "https://models-sif.s3.eu-north-1.amazonaws.com"
PORT = 8000


def _sh(command, timeout=None):
    """Run a shell command and return the completed process."""
    return subprocess.run(
        command, shell=True, capture_output=True, text=True, timeout=timeout
    )


def _wait_for_server(port=PORT, log="/content/serve.log", limit=300):
    """Block until the model server answers, or explain why it never did."""
    logfile = pathlib.Path(log)
    start = time.time()
    while time.time() - start < limit:
        text = logfile.read_text(errors="ignore") if logfile.exists() else ""
        if "CalledProcessError" in text or "Traceback (most recent call last)" in text:
            return False, "\n".join(text.splitlines()[-8:])
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/healthz", timeout=5) as r:
                if r.status == 200:
                    return True, f"{time.time() - start:.0f}s"
        except (urllib.error.URLError, OSError):
            time.sleep(3)
    return False, f"no response after {limit}s"


def _predict(smiles, port=PORT, batch=500):
    """Send SMILES to the running model, in batches, and return a list of dicts."""
    results = []
    for i in range(0, len(smiles), batch):
        chunk = smiles[i : i + batch]
        request = urllib.request.Request(
            f"http://127.0.0.1:{port}/run",
            data=json.dumps(chunk).encode(),
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(request, timeout=3600) as response:
            results.extend(json.loads(response.read()))
        print(f"  {min(i + batch, len(smiles))} / {len(smiles)} molecules")
    return results


_namespaces = 0
if os.path.exists("/proc/sys/user/max_user_namespaces"):
    _namespaces = int(open("/proc/sys/user/max_user_namespaces").read().strip() or 0)

if _namespaces == 0:
    print("FAILED: this runtime has user namespaces disabled, so containers cannot run here.")
elif shutil.which("apptainer"):
    print(f"Ready. Apptainer {_sh('apptainer --version').stdout.split()[-1]} is already installed.")
else:
    print("Installing Apptainer, about 40 seconds...")
    _sh("add-apt-repository -y ppa:apptainer/ppa && apt-get update -qq && apt-get install -y apptainer")
    if shutil.which("apptainer"):
        print(f"Ready. Installed Apptainer {_sh('apptainer --version').stdout.split()[-1]}.")
    else:
        print("FAILED: could not install Apptainer. Re-run this cell, or check the Colab image.")


In [ ]:
#@title 2. Choose a model and start it { display-mode: "form" }
#@markdown The model identifier from the Ersilia Model Hub, and the version of its image.
MODEL = "eos42ez"  #@param {type:"string"}
VERSION = "v1"  #@param {type:"string"}
#@markdown Downloading is the slow part: these images are several GB, and Colab discards them
#@markdown when the runtime is recycled.

_url = f"{BUCKET}/{MODEL}_{VERSION}.sif"
_sif = f"/content/{MODEL}_{VERSION}.sif"

# Check it exists before spending the download. This bucket returns 403, not 404, for a
# missing object, because it does not allow listing.
try:
    _head = urllib.request.Request(_url, method="HEAD")
    with urllib.request.urlopen(_head, timeout=30) as response:
        _size = int(response.headers["Content-Length"])
    print(f"Found {MODEL} {VERSION}: {_size / 1e9:.2f} GB")
except urllib.error.HTTPError:
    _size = None
    print(f"NOT FOUND: {MODEL} {VERSION}\n  Looked for {_url}\n"
          "  Check the identifier and the version, then run this cell again.")

if _size:
    if os.path.exists(_sif) and os.path.getsize(_sif) == _size:
        print("Already downloaded.")
    else:
        print("Downloading...")
        _done, _mark = 0, -5
        with urllib.request.urlopen(_url, timeout=60) as response, open(_sif, "wb") as handle:
            while True:
                _chunk = response.read(1 << 20)
                if not _chunk:
                    break
                handle.write(_chunk)
                _done += len(_chunk)
                _percent = int(_done * 100 / _size)
                if _percent >= _mark + 5:
                    print(f"  {_percent:3d}%  {_done / 2**30:.2f} / {_size / 1e9:.2f} GB")
                    _mark = _percent
        if os.path.getsize(_sif) != _size:
            print("FAILED: the download is incomplete. Run this cell again to resume.")
            _size = None

if _size:
    # The image's own entrypoint is unusable (it hardcodes a bundle path that does not exist
    # and ignores its arguments), so call the server directly at the bundle that is there.
    _bundle = _sh(
        f"unshare -r apptainer exec {_sif} "
        "sh -c 'ls -d ${ERSILIA_PATH:-/opt/ersilia}/bundles/*/ 2>/dev/null | head -1'"
    ).stdout.strip().rstrip("/")

    if not _bundle:
        print(f"FAILED: no model bundle inside {MODEL}_{VERSION}.sif. This image may not be "
              "an ersilia-pack build.")
    else:
        # The wrapper exits at once; the process holding the port is a run_uvicorn.py child.
        _sh("pkill -f 'ersilia_model_serve|run_uvicorn'")
        for _ in range(30):
            if not _sh(f"ss -lntH 'sport = :{PORT}'").stdout.strip():
                break
            time.sleep(1)

        pathlib.Path("/content/serve.log").unlink(missing_ok=True)
        print(f"Starting {MODEL}...")
        subprocess.Popen(
            f"unshare -r apptainer exec --bind /content:/content {_sif} "
            f"ersilia_model_serve --bundle_path {_bundle} --port {PORT} > /content/serve.log 2>&1",
            shell=True,
        )
        _ok, _detail = _wait_for_server()
        print(f"{MODEL} is running, ready after {_detail}." if _ok
              else f"FAILED: the model did not start.\n{_detail}")


In [ ]:
#@title 3. Upload a file and get predictions { display-mode: "form" }
#@markdown A `.csv` with a column of SMILES. Leave the column name blank to detect it
#@markdown automatically (a column called `smiles`, or the only column there is).
SMILES_COLUMN = ""  #@param {type:"string"}
import io

import pandas as pd
from google.colab import files

if "MODEL" not in globals():
    print("No model has been started yet. Run cells 1 and 2 first.")
else:
    _uploaded = files.upload()

    if not _uploaded:
        print("No file uploaded. Run this cell again.")
    else:
        _name = next(iter(_uploaded))
        try:
            _frame = pd.read_csv(io.BytesIO(_uploaded[_name]))
        except Exception as error:
            _frame = None
            print(f"Could not read {_name} as a CSV: {error}\n"
                  "Save the file as .csv and try again.")

        if _frame is not None:
            _column = SMILES_COLUMN.strip()
            if not _column:
                _matches = [c for c in _frame.columns if c.strip().lower() == "smiles"]
                if _matches:
                    _column = _matches[0]
                elif len(_frame.columns) == 1:
                    _column = _frame.columns[0]

            if _column not in _frame.columns:
                print(f"Could not tell which column holds the SMILES. Columns in {_name}: "
                      f"{list(_frame.columns)}\n"
                      "Type one of them into the box above and run this again.")
            else:
                _smiles = _frame[_column].astype(str).str.strip().tolist()
                print(f"{len(_smiles)} molecules from {_name}, column '{_column}'.")
                print("Predicting. Expect roughly a second per molecule, plus about "
                      "20 seconds of overhead per batch.")
                _predictions = _predict(_smiles)

                _result = pd.concat(
                    [_frame.reset_index(drop=True), pd.DataFrame(_predictions)], axis=1
                )
                _out = f"predictions_{MODEL}_{VERSION}.csv"
                _result.to_csv(_out, index=False)
                print(f"\nDone. Saved {_out}")
                display(_result.head(10))
                files.download(_out)


---

**If something goes wrong**

- *Cell 2 says NOT FOUND* — the identifier or the version is wrong. Versions look like `v1`.
- *Cell 2 says the model did not start* — read `/content/serve.log`; it holds the server's own error.
- *Cell 3 hangs* — big files are genuinely slow. A thousand molecules is around twenty minutes.
- *Everything stops working after a break* — Colab recycled the runtime and the image is gone.
  Run cell 1, then cell 2 again.

Only one model runs at a time: cell 2 stops whatever was running before it starts the new one.
